## Pretrained models and Transfer Learning
- Can we use a neural network **trained on one dataset and adapt it** to classifying different images without full training process? - **Transfer Learning**
-  In **transfer learning**, we typically **start with a pre-trained model**, which has been trained on some large image dataset, such as **ImageNet**. Those models can already do a good job extracting different features from generic images, and in many cases just building a classifier on top of those extracted features can yield a good result.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchinfo import summary
import numpy as np
import os

from pytorchcv.model_provider import get_model

In [ ]:
def train(model, dataloader, criterion, optimizer, device):
    model.train()
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

def check_image_dir(dir_name):
    classes = os.listdir(dir_name)
    fig, axs = plt.subplots(1, len(classes), figsize=(12, 4))
    
    # Handle single class directory edge case
    if len(classes) == 1:
        axs = [axs]

    for i, cls in enumerate(classes):
        cls_dir = os.path.join(dir_name, cls)
        if os.path.isdir(cls_dir):
            images = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            if images:
                img_path = os.path.join(cls_dir, images[0])
                img = Image.open(img_path)
                axs[i].imshow(img)
                axs[i].set_title(f"{cls}\n({len(images)} images)")
                axs[i].axis('off')
    
    plt.tight_layout()
    plt.show()

def display_dataset(dataloader, class_names=['Cat', 'Dog'], num_images=8):
    # 1. Grab a single batch of data from the dataloader
    dataiter = iter(dataloader)
    images, labels = next(dataiter)
    
    # Ensure we don't try to show more images than the batch contains
    num_to_show = min(num_images, images.size(0))
    
    # 2. Set up the matplotlib figure
    fig, axes = plt.subplots(1, num_to_show, figsize=(15, 3))
    
    # Handle the case where we only show 1 image (axes becomes a single object, not an array)
    if num_to_show == 1:
        axes = [axes]
        
    for i in range(num_to_show):
        # 3. Convert PyTorch tensor to NumPy array
        img = images[i].numpy()
        
        # 4. Rearrange dimensions from Channels-First (C, H, W) to Channels-Last (H, W, C)
        img = np.transpose(img, (1, 2, 0))
        
        # 5. Clip values between 0 and 1 to prevent matplotlib warnings 
        # (Useful if you applied normalization transforms)
        img = np.clip(img, 0, 1)
        
        # 6. Display image and title
        axes[i].imshow(img)
        
        # Assign the correct class name based on the label tensor (0 = Cat, 1 = Dog)
        label_idx = labels[i].item()
        title = class_names[label_idx] if class_names else f"Class: {label_idx}"
        
        axes[i].set_title(title)
        axes[i].axis('off') # Hide axes ticks for cleaner look
        
    plt.tight_layout()
    plt.show()

std_normalize = transforms.Normalize(
    mean = [0.485, 0.456, 0.406],
    std  = [0.229, 0.224, 0.225]
) # Applying std_normalize transform to bring images to the range expected by pre-trained VGG network.

trans = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    std_normalize
])

dataset = torchvision.datasets.ImageFolder('data/pet_images', transform = trans)
train_set, test_set = torch.utils.data.random_split(dataset, [20000, len(dataset) - 20000])

### Loading dataset

In [ ]:
import zipfile
if not os.path.exists('../../data/PetImages'):
    with zipfile.ZipFile('../../data/dogs-vs-cats.zip', 'r') as zip_ref:
        zip_ref.extractall('../../data')

with zipfile.ZipFile('../../data/test1.zip', 'r') as zip_ref:
        zip_ref.extractall('../../data')
with zipfile.ZipFile('../../data/train.zip', 'r') as zip_ref:
        zip_ref.extractall('../../data')

In [ ]:
import shutil

# 1. Define your source directory
# Update this if your path requires navigating up or down directories (e.g., '../../train')
source_dir = '../../data/train' 

# 2. Define the target directories
cat_dir = os.path.join(source_dir, 'Cat')
dog_dir = os.path.join(source_dir, 'Dog')

# 3. Create the 'Cat' and 'Dog' folders if they don't exist already
os.makedirs(cat_dir, exist_ok=True)
os.makedirs(dog_dir, exist_ok=True)

# 4. Loop through every file in the source folder
for filename in os.listdir(source_dir):
    
    # Construct the full file path
    file_path = os.path.join(source_dir, filename)
    
    # Skip directories (so we don't try to move the Cat/Dog folders we just made)
    if os.path.isdir(file_path):
        continue

    # 5. Move the files based on their prefix
    if filename.lower().startswith('cat'):
        shutil.move(file_path, os.path.join(cat_dir, filename))
    elif filename.lower().startswith('dog'):
        shutil.move(file_path, os.path.join(dog_dir, filename))

print("Organizing complete! Check your 'train' folder.")

In [ ]:
std_normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
trans = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(), 
        std_normalize])
dataset = torchvision.datasets.ImageFolder('../../data/PetImages',transform=trans)
trainset, testset = torch.utils.data.random_split(dataset,[20000,len(dataset)-20000])

temp_loader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True)
display_dataset(temp_loader)

### Pre-trained model
- There are many different pre-trained models available inside *torchvision* module. Let's see how simpleset **VGG-16** model can be loaded and used:

In [ ]:
vgg = torchvision.models.vgg16(pretrained=True)
sample_image = dataset[0][0].unsqueeze(0)
res = vgg(sample_image)
print(res[0].argmax()) # The result we received is a number of an *ImageNet* class

#Output: tensor(282)

### GPU Computations
- Deep neural networks *require a lot of computational* power to run. It makes sense to use **GPU acceleration**, if it is available. 

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Doing computations on device = {}', format(device))

vgg.to(device)
sample_image = sample_image.to(device)
res = vgg(sample_image)
print(res.argmax())

#Output: tensor(282, device='cuda:0')

### Extracting VGG features
- In this 'feature extractor' step, we can use **vgg.features** method:

In [ ]:
res = vgg.features(sample_image).cpu()
print(res.size()) #Output: torch.size([1, 512, 7, 7])

The dimension of feature tensor is 512x7x7.
Let's manually take some portion of images (800 in our case), and pre-compute their feature vectors. We will store the result in 1 big tensor called **feature_tensor**, and also labels into **label_tensor**

In [ ]:
bs = 8
dl = torch.utils.data.DataLoader(dataset, batch_size = bs, shuffle=True)

num = bs * 100
feature_tensor = torch.zeros(num, 512 * 7 * 7).to(device)
label_tensor = torch.zeros(num).to(device)

i = 0
for x, l in dl:
    with torch.no_grad():
        f = vgg.features(x.to(device))
        feature_tensor[i:i+bs] = f.view(bs, -1)
        label_tensor[i:i+bs] = l
        i += bs
        print('.', end = '')

        if i >= num:
            break

In [ ]:
vgg_dataset = torch.utils.data.TensorDataset(feature_tensor, label_tensor.to(torch.long)) # vgg_dataset takes data from feature_tensor and label_tensor, split it into train/test set using random_split
train_ds, test_ds = torch.utils.data.random_split(vgg_dataset, [700, 100])

train_loader = torch.utils.data.DataLoader(train_ds, batch_size = 32)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size = 32)

net = torch.nn.Sequential(
    torch.nn.Linear(512 * 7 * 7, 2),
    torch.nn.LogSoftmax()
).to(device)

hist = train(net, train_loader, test_loader)

print(vgg)

## Transfer Learning using VGG16 network

In VGG16 structure contains:
- Feature extractor (feature), comprised of a number of convolutional and pooling layers
- Average pooling layer (avgpool)
- Final classifier, consisting of several dense layers, which turns **25088 input features** into **1000 classes** (which is the number of classes in **ImageNet**)

To finetune this end-to-end model to classify our dataset, we need to:
- **Replace the Final classifier** to match with the number of our dataset's classes.
- **Freeze weights of convolutioanl feature extractor**

In [ ]:
#Adjust the classifier
vgg.classifier = torch.nn.Linear(512 * 7 * 7, 2).to(device)

#Freeze weights of feature extractor
for x in vgg.features.parameters():
    x.requires_grad = False

summary(vgg, (1, 3, 244, 244))

In [ ]:
train_set, test_set = torch.utils.data.random_split(dataset, [20000, len(dataset) - 20000])
train_loader = torch.utils.data.DataLoader(train_set, batch_size = 16)
test_loader = torch.utils.data.DataLoader(test_set, batch_size = 16)

train_long(vgg, train_loader, test_loader, loss_fn = torch.nn.CrossEntropyLoss(), epochs = 1, print_freq = 90)


In [ ]:
torch.save(vgg,'data/cats_dogs.pth')
vgg = torch.load('data/cats_dogs.pth')

### Fine-tuning transfer learning
 If your objects visually differ from ordinary ImageNet images, this combination of features might not work best. Thus it makes sense to start training convolutional layers as well.

In [ ]:
for x in vgg.features.parameters():
    x.requires_grad = True

train_long(vgg, train_loader, test_loader, loss_fn = torch.nn.CrossEntropyLoss(), epochs = 1, print_freq = 90, lr = 0.0001)